In [1]:
import networkx as nx
import numpy as np
import pandas as pd

SEEDS = [42, 7, 13]
NUM_SAMPLES = 5000
data = []

for seed in SEEDS:
    np.random.seed(seed)
    G = nx.erdos_renyi_graph(50, 0.15, seed=seed)
    for u, v in G.edges():
        G[u][v]['latency'] = round(np.random.uniform(1, 10), 2)

    deg = nx.degree_centrality(G)
    bet = nx.betweenness_centrality(G)
    clo = nx.closeness_centrality(G)
    nodes = list(G.nodes())

    for _ in range(NUM_SAMPLES // len(SEEDS)):
        src = np.random.choice(nodes)
        dst = np.random.choice([n for n in nodes if n != src])
        try:
            path = nx.shortest_path(G, src, dst, weight='latency')
            if len(path) < 2:
                continue
            current = path[0]
            next_hop = path[1]
            neighbors = list(G.neighbors(current))
            traffic_load = np.random.uniform(0, 1)
            data.append({
                'node': current,
                'degree': deg[current],
                'betweenness': bet[current],
                'closeness': clo[current],
                'traffic_load': traffic_load,
                'dst': dst,
                'optimal_next_hop': next_hop,
                'seed': seed
            })
        except nx.NetworkXNoPath:
            continue

df = pd.DataFrame(data)
df.to_csv('../results/routing_dataset.csv', index=False)
print("Dataset saved:", len(df), "samples")
print(df.head())

Dataset saved: 4998 samples
   node    degree  betweenness  closeness  traffic_load  dst  \
0     0  0.204082     0.046423   0.505155      0.341066   11   
1    24  0.061224     0.002811   0.395161      0.541448   22   
2    41  0.183673     0.042692   0.510417      0.228550   34   
3    15  0.183673     0.031542   0.500000      0.982168   26   
4    48  0.102041     0.012431   0.457944      0.996254    1   

   optimal_next_hop  seed  
0                20    42  
1                 9    42  
2                 6    42  
3                44    42  
4                20    42  
